# Tema de Programare: Selecția Statistică a Caracteristicilor

Bine ai venit la tema de programare despre Selecția Statistică a Caracteristicilor!

Selecția caracteristicilor este procesul de identificare și păstrare doar a celor mai relevante caracteristici pentru modelul tău. Acest lucru reduce zgomotul, accelerează antrenarea și îmbunătățește adesea generalizarea.

**Ce vei face în această temă:**

* **Metode Filtru** – Aplici pragul de varianță, selecția bazată pe corelație și selecția pe baza informației mutuale
* **Selecția prin Teste Statistice** – Folosești `SelectKBest` cu `f_classif`, `f_regression` și `chi2` pentru tipuri diferite de probleme
* **Eliminarea Bazată pe Corelație** – Elimini în mod sistematic caracteristicile foarte corelate (redundante)
* **Pipeline de Selecție a Caracteristicilor** – Construiești un flux complet: scalare → selecție → model
* **Implementare de la Zero** – Implementezi manual statistica F ANOVA

Să începem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu excepția celor în care trebuie să trimiți soluțiile sau când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga și nu modifica niciun cod în afara acestor comentarii**.

* Poți adăuga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, așa că nu te baza pe celulele create de tine pentru codul soluției; folosește spațiile oferite în notebook.

* Evită folosirea variabilelor globale, cu excepția situațiilor absolut necesare. Evaluatorul îți testează codul într-un mediu izolat fără să ruleze toate celulele de la început. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salvează-l mai întâi apăsând pe iconița 💾 din stânga sus a paginii, apoi apasă pe butonul `Submit assignment` din dreapta sus.
---


## Cuprins
- [Importuri](#imports)
- [1 - Metode Filtru](#1)
    - [1.1 - Încarcă Datele](#1-1)
    - [1.2 - Prag de Varianță](#1-2)
        - **[Exercițiul 1 - apply_variance_threshold](#ex-1)**
    - [1.3 - Vizualizează Varianțele Caracteristicilor](#1-3)
        - **[Exercițiul 2 - plot_variance_scores](#ex-2)**
    - [1.4 - Selecția pe baza Informației Mutuale](#1-4)
        - **[Exercițiul 3 - select_with_mutual_information](#ex-3)**
- [2 - Selecția prin Teste Statistice](#2)
    - [2.1 - Testul F pentru Clasificare](#2-1)
        - **[Exercițiul 4 - select_with_f_classif](#ex-4)**
    - [2.2 - Testul Chi-Pătrat](#2-2)
        - **[Exercițiul 5 - select_with_chi2](#ex-5)**
    - [2.3 - Testul F pentru Regresie](#2-3)
        - **[Exercițiul 6 - select_for_regression](#ex-6)**
    - [2.4 - Compară Metodele de Selecție](#2-4)
        - **[Exercițiul 7 - compare_selection_methods](#ex-7)**
- [3 - Eliminarea Bazată pe Corelație](#3)
    - [3.1 - Găsește Perechi Corelate](#3-1)
        - **[Exercițiul 8 - find_correlated_pairs](#ex-8)**
    - [3.2 - Elimină Caracteristicile Corelate](#3-2)
        - **[Exercițiul 9 - remove_correlated_features](#ex-9)**
- [4 - Pipeline de Selecție a Caracteristicilor](#4)
    - [4.1 - Construiește Pipeline-ul Complet](#4-1)
        - **[Exercițiul 10 - create_selection_pipeline](#ex-10)**
- [5 - Implementare de la Zero](#5)
    - [5.1 - Statistica F ANOVA](#5-1)
        - **[Bonus - compute_f_statistic_from_scratch](#ex-bonus)**
- [6 - Raport de Selecție a Caracteristicilor](#6)


<a name='imports'></a>
## Importuri


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
%matplotlib inline

warnings.filterwarnings('ignore', message='.*FigureCanvasAgg is non-interactive.*')

from sklearn.datasets import make_classification, make_regression, load_wine
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, SelectPercentile,
    f_classif, f_regression, chi2,
    mutual_info_classif, mutual_info_regression
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('Bibliotecile au fost încărcate cu succes!')


<a name='1'></a>
## 1 - Metode Filtru

Metodele filtru evaluează fiecare caracteristică independent de orice model. Ele sunt cea mai rapidă și cea mai generală familie de tehnici de selecție a caracteristicilor.

În această secțiune vei aplica trei metode filtru:
1. **Pragul de Varianță** – elimină caracteristicile cu varianță aproape de zero (caracteristici constante)
2. **Informația Mutuală** – punctează caracteristicile în funcție de dependența lor statistică față de țintă
3. **Vizualizare** – reprezintă distribuțiile varianțelor și scorurilor


<a name='1-1'></a>
### 1.1 - Încarcă Datele

Vom lucra cu trei seturi de date:
- Un set de date **sintetic pentru clasificare** (multe caracteristici redundante) — folosit în exercițiile de clasificare
- Un set de date **Wine** — folosit pentru analiza corelației
- Un set de date **sintetic pentru regresie** — folosit în exercițiile de regresie


In [ ]:
# === SET DE DATE PENTRU CLASIFICARE ===
X_clf, y_clf = make_classification(
    n_samples=600,
    n_features=30,
    n_informative=8,
    n_redundant=12,
    n_repeated=2,
    n_classes=3,
    random_state=42
)
FEATURE_NAMES_CLF = [f'feature_{i}' for i in range(X_clf.shape[1])]
print(f'Set de date pentru clasificare: {X_clf.shape[0]} eșantioane, {X_clf.shape[1]} caracteristici, {len(set(y_clf))} clase')

# === SET DE DATE PENTRU REGRESIE ===
X_reg, y_reg = make_regression(
    n_samples=400,
    n_features=20,
    n_informative=6,
    noise=0.3,
    random_state=42
)
FEATURE_NAMES_REG = [f'feature_{i}' for i in range(X_reg.shape[1])]
print(f'Set de date pentru regresie:     {X_reg.shape[0]} eșantioane, {X_reg.shape[1]} caracteristici')

# === SETUL DE DATE WINE (pentru analiza corelației) ===
wine = load_wine()
X_wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine = wine.target
print(f'Setul de date Wine:              {X_wine_df.shape[0]} eșantioane, {X_wine_df.shape[1]} caracteristici')


In [ ]:
# Standardizează seturile de date (necesar pentru majoritatea testelor statistice)
scaler_clf = StandardScaler()
X_clf_scaled = scaler_clf.fit_transform(X_clf)

scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

# Versiune scalată nenegativă pentru chi2
mms = MinMaxScaler()
X_clf_nonneg = mms.fit_transform(X_clf)

print('Datele au fost standardizate cu succes.')


<a name='1-2'></a>
### 1.2 - Prag de Varianță

<a name='ex-1'></a>
**Exercițiul 1 - apply_variance_threshold**

Implementează o funcție care aplică `VarianceThreshold` pe o matrice de caracteristici.

**Instrucțiuni:**
- Creează un selector `VarianceThreshold` cu `threshold`-ul dat
- Aplică `fit` și `transform` pe `X`
- Returnează obiectul selector antrenat și array-ul transformat

**Indiciu:** Folosește `sklearn.feature_selection.VarianceThreshold`


In [ ]:
def apply_variance_threshold(X, threshold=0.0):
    """
    Aplică filtrul pe baza pragului de varianță pentru a elimina
    caracteristicile cu varianță mică.

    Args:
        X: Array de caracteristici de formă (m, n)
        threshold: Varianța minimă necesară pentru a păstra o caracteristică (implicit 0.0)

    Returns:
        selector: Obiect VarianceThreshold antrenat
        X_filtered: Array transformat, din care au fost eliminate caracteristicile cu varianță mică
    """
    ### ÎNCEPUT DE COD AICI ### (~3 lines)
    selector = None
    X_filtered = None
    ### SFÂRȘIT DE COD AICI ###

    return selector, X_filtered


In [ ]:
# Test pentru apply_variance_threshold
# Creează un set de date cu o coloană constantă și una cu varianță aproape de zero
test_X = np.column_stack([
    np.random.randn(100),      # caracteristică normală, varianță mare
    np.zeros(100),             # caracteristică constantă, varianță = 0
    np.random.randn(100) * 0.001,  # varianță aproape de zero
    np.random.randn(100) * 2,  # varianță mare
])

sel, X_t = apply_variance_threshold(test_X, threshold=0.01)

assert sel is not None, 'selector must not be None'
assert X_t is not None, 'X_filtered must not be None'
assert X_t.shape[1] == 2, f'Expected 2 features after threshold, got {X_t.shape[1]}'

print('Exercițiul 1 a fost rezolvat corect!')
print(f'Caracteristici inițiale: {test_X.shape[1]}  |  După prag: {X_t.shape[1]}')


In [ ]:
unittests.exercise_1(apply_variance_threshold)

<a name='1-3'></a>
### 1.3 - Vizualizează Varianțele Caracteristicilor

<a name='ex-2'></a>
**Exercițiul 2 - plot_variance_scores**

Creează un grafic cu bare orizontale pentru varianțele caracteristicilor, evidențiind ce caracteristici se află sub prag.

**Instrucțiuni:**
- Sortează caracteristicile după varianță în ordine descrescătoare
- Afișează primele `top_n` caracteristici
- Desenează o linie verticală roșie punctată la valoarea `threshold`
- Colorează barele: portocaliu dacă sunt sub prag, steelblue dacă sunt peste
- Adaugă eticheta pentru axa x `"Varianță"`, pentru axa y `"Caracteristică"` și un titlu


In [ ]:
def plot_variance_scores(variances, feature_names, threshold=0.0, top_n=20):
    """
    Reprezintă varianțele caracteristicilor ca diagramă cu bare orizontale.

    Args:
        variances: Array cu valorile varianței, una pentru fiecare caracteristică
        feature_names: Listă/array cu numele caracteristicilor
        threshold: Linia prag care trebuie desenată
        top_n: Numărul de caracteristici care vor fi afișate (primele n după varianță)

    Returns:
        None (afișează graficul)
    """
    ### ÎNCEPUT DE COD AICI ### (~12 lines)

    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test pentru plot_variance_scores — rulează această celulă pentru a vedea graficul
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.0)
vt.fit(X_clf_scaled)
feature_variances = vt.variances_

plot_variance_scores(feature_variances, FEATURE_NAMES_CLF, threshold=0.5, top_n=30)
print('Exercițiul 2: graficul este afișat mai sus')


In [ ]:
unittests.exercise_2(plot_variance_scores)

<a name='1-4'></a>
### 1.4 - Selecția pe baza Informației Mutuale

<a name='ex-3'></a>
**Exercițiul 3 - select_with_mutual_information**

Selectează primele `k` caracteristici folosind informația mutuală.

**Instrucțiuni:**
- Folosește `SelectKBest` cu `mutual_info_classif` pentru probleme de clasificare sau `mutual_info_regression` pentru regresie
- Aplică `fit` pe `(X, y)` și transformă `X`
- Returnează selectorul și array-ul cu caracteristicile selectate

**Indiciu:** `task_type` va fi fie `'classification'`, fie `'regression'`


In [ ]:
def select_with_mutual_information(X, y, k, task_type='classification'):
    """
    Selectează cele mai bune k caracteristici folosind informația mutuală.

    Args:
        X: Array de caracteristici
        y: Array țintă
        k: Numărul de caracteristici de selectat
        task_type: 'classification' sau 'regression'

    Returns:
        selector: Obiect SelectKBest antrenat
        X_selected: Array transformat cu k caracteristici
    """
    ### ÎNCEPUT DE COD AICI ### (~6 lines)
    selector = None
    X_selected = None
    ### SFÂRȘIT DE COD AICI ###

    return selector, X_selected


In [ ]:
# Test pentru select_with_mutual_information
k = 8
sel_mi, X_mi = select_with_mutual_information(X_clf_scaled, y_clf, k=k)

assert sel_mi is not None, 'selector must not be None'
assert X_mi is not None, 'X_selected must not be None'
assert X_mi.shape == (X_clf_scaled.shape[0], k), \
    f'Expected shape ({X_clf_scaled.shape[0]}, {k}), got {X_mi.shape}'

selected_names = [FEATURE_NAMES_CLF[i] for i in range(len(FEATURE_NAMES_CLF)) if sel_mi.get_support()[i]]
print('Exercițiul 3 a fost rezolvat corect!')
print(f'Au fost selectate {k} caracteristici: {selected_names}')


In [ ]:
unittests.exercise_3(select_with_mutual_information)

<a name='2'></a>
## 2 - Selecția prin Teste Statistice

Testele statistice îți oferă un **scor și o valoare p** pentru fiecare caracteristică, măsurând cât de puternic se leagă aceasta de țintă.

Regula cheie: **alege testul care se potrivește tipului tău de problemă:**
- Clasificare + caracteristici continue → `f_classif`
- Clasificare + caracteristici nenegative → `chi2`
- Regresie + caracteristici continue → `f_regression`


<a name='2-1'></a>
### 2.1 - Testul F pentru Clasificare

<a name='ex-4'></a>
**Exercițiul 4 - select_with_f_classif**

Selectează primele `k` caracteristici pentru o problemă de clasificare folosind testul F ANOVA.

**Instrucțiuni:**
- Folosește `SelectKBest` cu `f_classif`
- Construiește și returnează un **DataFrame cu scoruri** care conține coloanele: `'feature'`, `'F_score'`, `'p_value'`, sortat descrescător după `'F_score'`
- Returnează selectorul antrenat, array-ul cu caracteristicile selectate și DataFrame-ul cu scoruri


In [ ]:
def select_with_f_classif(X, y, k, feature_names):
    """
    Selectează cele mai bune k caracteristici pentru clasificare folosind testul F ANOVA.

    Args:
        X: Array de caracteristici (valori continue)
        y: Ținta de clasificare
        k: Numărul de caracteristici de selectat
        feature_names: Lista numelor caracteristicilor

    Returns:
        selector: Obiect SelectKBest antrenat
        X_selected: Array transformat cu k caracteristici
        scores_df: DataFrame cu coloanele ['feature','F_score','p_value'],
                   sortat descrescător după F_score
    """
    ### ÎNCEPUT DE COD AICI ### (~8 lines)
    selector = None
    X_selected = None
    scores_df = None
    ### SFÂRȘIT DE COD AICI ###

    return selector, X_selected, scores_df


In [ ]:
# Test pentru select_with_f_classif
k = 10
sel_f, X_f, scores_f = select_with_f_classif(X_clf_scaled, y_clf, k, FEATURE_NAMES_CLF)

assert sel_f is not None, 'selector must not be None'
assert X_f.shape == (X_clf_scaled.shape[0], k), f'Expected shape ({X_clf_scaled.shape[0]}, {k})'
assert scores_f is not None, 'scores_df must not be None'
assert set(scores_f.columns) >= {'feature', 'F_score', 'p_value'}, 'Missing columns in scores_df'
assert list(scores_f['F_score']) == sorted(scores_f['F_score'], reverse=True), 'scores_df not sorted'

print('Exercițiul 4 a fost rezolvat corect!')
print(f'\nPrimele 10 caracteristici după F-score:')
print(scores_f.head(10).to_string(index=False))


In [ ]:
unittests.exercise_4(select_with_f_classif)

<a name='2-2'></a>
### 2.2 - Testul Chi-Pătrat

<a name='ex-5'></a>
**Exercițiul 5 - select_with_chi2**

Selectează primele `k` caracteristici folosind testul chi-pătrat.

**Important:** `chi2` necesită ca toate valorile caracteristicilor să fie **nenegative**. Folosește `X_clf_nonneg` (deja scalat cu `MinMaxScaler`).

**Instrucțiuni:**
- Folosește `SelectKBest` cu `chi2`
- Intrarea `X_nonneg` trebuie să aibă deja valori nenegative
- Returnează selectorul antrenat și array-ul cu caracteristicile selectate


In [ ]:
def select_with_chi2(X_nonneg, y, k):
    """
    Selectează cele mai bune k caracteristici folosind testul chi-pătrat.

    Args:
        X_nonneg: Array de caracteristici cu valori nenegative (de ex., după MinMaxScaler)
        y: Ținta de clasificare
        k: Numărul de caracteristici de selectat

    Returns:
        selector: Obiect SelectKBest antrenat
        X_selected: Array transformat cu k caracteristici
    """
    ### ÎNCEPUT DE COD AICI ### (~3 lines)
    selector = None
    X_selected = None
    ### SFÂRȘIT DE COD AICI ###

    return selector, X_selected


In [ ]:
# Test pentru select_with_chi2
k = 10
sel_chi2, X_chi2 = select_with_chi2(X_clf_nonneg, y_clf, k)

assert sel_chi2 is not None, 'selector must not be None'
assert X_chi2 is not None, 'X_selected must not be None'
assert X_chi2.shape == (X_clf_nonneg.shape[0], k), \
    f'Expected shape ({X_clf_nonneg.shape[0]}, {k}), got {X_chi2.shape}'
assert np.all(X_chi2 >= 0), 'All values should be non-negative (chi2 requires this)'

print('Exercițiul 5 a fost rezolvat corect!')
print(f'chi2 a selectat {k} caracteristici din {X_clf_nonneg.shape[1]}')


In [ ]:
unittests.exercise_5(select_with_chi2)

<a name='2-3'></a>
### 2.3 - Testul F pentru Regresie

<a name='ex-6'></a>
**Exercițiul 6 - select_for_regression**

Selectează primele `k` caracteristici pentru o problemă de **regresie** folosind testul F.

**Instrucțiuni:**
- Folosește `SelectKBest` cu `f_regression`
- Returnează selectorul antrenat, array-ul cu caracteristicile selectate și un DataFrame cu scoruri cu coloanele `['feature', 'F_score', 'p_value']`, sortat descrescător după F_score


In [ ]:
def select_for_regression(X, y, k, feature_names):
    """
    Selectează cele mai bune k caracteristici pentru regresie folosind testul F.

    Args:
        X: Array de caracteristici (valori continue)
        y: Țintă continuă de regresie
        k: Numărul de caracteristici de selectat
        feature_names: Lista numelor caracteristicilor

    Returns:
        selector: Obiect SelectKBest antrenat
        X_selected: Array transformat cu k caracteristici
        scores_df: DataFrame cu coloanele ['feature','F_score','p_value'],
                   sortat descrescător după F_score
    """
    ### ÎNCEPUT DE COD AICI ### (~8 lines)
    selector = None
    X_selected = None
    scores_df = None
    ### SFÂRȘIT DE COD AICI ###

    return selector, X_selected, scores_df


In [ ]:
# Test pentru select_for_regression
k = 6
sel_reg, X_reg_s, scores_reg = select_for_regression(X_reg_scaled, y_reg, k, FEATURE_NAMES_REG)

assert sel_reg is not None, 'selector must not be None'
assert X_reg_s.shape == (X_reg_scaled.shape[0], k)
assert scores_reg is not None
assert list(scores_reg['F_score']) == sorted(scores_reg['F_score'], reverse=True)

print('Exercițiul 6 a fost rezolvat corect!')
print(f'\nPrimele caracteristici pentru regresie:')
print(scores_reg.head(k).to_string(index=False))


In [ ]:
unittests.exercise_6(select_for_regression)

<a name='2-4'></a>
### 2.4 - Compară Metodele de Selecție

<a name='ex-7'></a>
**Exercițiul 7 - compare_selection_methods**

Compară clasamentele caracteristicilor obținute cu `f_classif` și `mutual_info_classif`. Cele două metode pot să nu fie de acord uneori — este important să înțelegi când și de ce.

**Instrucțiuni:**
- Calculează scorurile folosind atât `f_classif`, cât și `mutual_info_classif` cu `k='all'`
- Atribuie ranguri (1 = cel mai bun) fiecărei caracteristici pentru fiecare metodă
- Returnează un DataFrame cu coloanele: `'feature'`, `'f_classif_score'`, `'mi_score'`, `'f_rank'`, `'mi_rank'`, `'rank_diff'` (diferența absolută dintre ranguri), sortat după `'f_rank'`


In [ ]:
def compare_selection_methods(X, y, feature_names):
    """
    Compară clasamentele caracteristicilor obținute cu f_classif și mutual_info_classif.

    Args:
        X: Array de caracteristici
        y: Ținta de clasificare
        feature_names: Lista numelor caracteristicilor

    Returns:
        comparison_df: DataFrame cu coloanele:
            ['feature', 'f_classif_score', 'mi_score',
             'f_classif_rank', 'mi_rank', 'rank_diff']
            sortat crescător după f_classif_rank
    """
    ### ÎNCEPUT DE COD AICI ### (~15 lines)
    comparison_df = None
    ### SFÂRȘIT DE COD AICI ###

    return comparison_df


In [ ]:
# Test pentru compare_selection_methods
comparison = compare_selection_methods(X_clf_scaled, y_clf, FEATURE_NAMES_CLF)

assert comparison is not None, 'comparison_df must not be None'
expected_cols = {'feature', 'f_classif_score', 'mi_score', 'f_rank', 'mi_rank', 'rank_diff'}
assert expected_cols.issubset(set(comparison.columns)), f'Missing columns: {expected_cols - set(comparison.columns)}'
assert len(comparison) == len(FEATURE_NAMES_CLF), 'DataFrame should have one row per feature'

print('Exercițiul 7 a fost rezolvat corect!')
print('\nCaracteristicile pentru care metodele diferă cel mai mult (cea mai mare diferență de rang):')
print(comparison.nlargest(5, 'rank_diff')[['feature','f_rank','mi_rank','rank_diff']].to_string(index=False))


In [ ]:
unittests.exercise_7(compare_selection_methods)

<a name='3'></a>
## 3 - Eliminarea Bazată pe Corelație

Testele statistice îți spun ce caracteristici se leagă de **țintă**. Dar nu îți spun când două caracteristici se leagă **între ele** (redundanță).

Eliminarea bazată pe corelație găsește perechi de caracteristici foarte corelate și o elimină pe cea mai puțin relevantă din fiecare pereche.


In [ ]:
# Vizualizează matricea de corelație a caracteristicilor din setul Wine
corr_matrix = X_wine_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Setul de date Wine — Matricea de corelație a caracteristicilor (triunghiul inferior)')
plt.tight_layout()
plt.show()
print('Valorile mari (aproape de ±1) indică perechi de caracteristici redundante.')


<a name='3-1'></a>
### 3.1 - Găsește Perechi Corelate

<a name='ex-8'></a>
**Exercițiul 8 - find_correlated_pairs**

Identifică toate perechile de caracteristici a căror corelație Pearson în valoare absolută depășește un prag.

**Instrucțiuni:**
- Calculează matricea corelațiilor absolute pornind de la `X_df`
- Privește doar **triunghiul superior** pentru a evita perechile duplicate
- Colectează tuple de forma `(feature_a, feature_b, correlation_value)` pentru toate perechile peste `threshold`
- Returnează lista sortată descrescător după corelație


In [ ]:
def find_correlated_pairs(X_df, threshold=0.85):
    """
    Găsește toate perechile de caracteristici cu corelație absolută peste prag.

    Args:
        X_df: DataFrame pandas cu caracteristici
        threshold: Corelația absolută minimă pentru a semnala o pereche

    Returns:
        correlated_pairs: Listă de tuple (feature_a, feature_b, corr_value)
                          sortată descrescător după corr_value
    """
    ### ÎNCEPUT DE COD AICI ### (~10 lines)
    correlated_pairs = []
    ### SFÂRȘIT DE COD AICI ###

    return correlated_pairs


In [ ]:
# Test pentru find_correlated_pairs pe setul de date Wine
pairs = find_correlated_pairs(X_wine_df, threshold=0.75)

assert isinstance(pairs, list), 'Return value must be a list'
assert len(pairs) > 0, 'Expected at least one correlated pair in wine dataset above 0.75'
for p in pairs:
    assert len(p) == 3, 'Each element must be a 3-tuple (feat_a, feat_b, corr)'
    assert abs(p[2]) >= 0.75, f'Correlation {p[2]:.3f} below threshold'

# Verifică sortarea descrescătoare
corr_values = [abs(p[2]) for p in pairs]
assert corr_values == sorted(corr_values, reverse=True), 'Pairs must be sorted by correlation descending'

print('Exercițiul 8 a fost rezolvat corect!')
print(f'\nAu fost găsite {len(pairs)} perechi corelate (|r| > 0.75):')
for feat_a, feat_b, corr in pairs[:5]:
    print(f'  {feat_a[:20]:20s}  <->  {feat_b[:20]:20s}  r = {corr:.3f}')


In [ ]:
unittests.exercise_8(find_correlated_pairs)

<a name='3-2'></a>
### 3.2 - Elimină Caracteristicile Corelate

<a name='ex-9'></a>
**Exercițiul 9 - remove_correlated_features**

Algoritm greedy: pentru fiecare pereche corelată, elimină caracteristica cu **scorul de relevanță mai mic** față de țintă.

**Instrucțiuni:**
- Calculează matricea corelațiilor absolute și privește doar triunghiul superior
- Pentru fiecare pereche cu corelație > prag, adaugă în setul de eliminare caracteristica având valoarea `target_scores` **mai mică**
- Returnează două liste: caracteristicile păstrate și caracteristicile eliminate


In [ ]:
def remove_correlated_features(X_df, target_scores, threshold=0.85):
    """
    Eliminare greedy a caracteristicilor corelate.
    Păstrează caracteristica cu scorul mai mare de relevanță față de țintă din fiecare pereche.

    Args:
        X_df: DataFrame pandas cu caracteristici
        target_scores: dict {feature_name: relevance_score}
        threshold: Pragul pentru corelația absolută

    Returns:
        features_to_keep: Listă cu numele caracteristicilor care trebuie păstrate
        features_removed: Listă cu numele caracteristicilor eliminate
    """
    ### ÎNCEPUT DE COD AICI ### (~15 lines)
    features_to_keep = []
    features_removed = []
    ### SFÂRȘIT DE COD AICI ###

    return features_to_keep, features_removed


In [ ]:
# Test pentru remove_correlated_features pe setul de date Wine
# Mai întâi obține scorurile F ca metrică de relevanță
f_sel = SelectKBest(f_classif, k='all')
f_sel.fit(X_wine_df.values, y_wine)
wine_scores = dict(zip(X_wine_df.columns, f_sel.scores_))

threshold = 0.75
kept, removed = remove_correlated_features(X_wine_df, wine_scores, threshold=threshold)

assert isinstance(kept, list)
assert isinstance(removed, list)
assert len(kept) + len(removed) == X_wine_df.shape[1], 'kept + removed must equal total features'
assert len(set(kept) & set(removed)) == 0, 'A feature cannot be both kept and removed'

# Verifică să nu mai existe nicio pereche peste prag
X_kept = X_wine_df[kept]
corr_kept = X_kept.corr().abs()
upper = corr_kept.where(np.triu(np.ones(corr_kept.shape), k=1).astype(bool))
max_corr = upper.stack().max() if not upper.stack().empty else 0
assert max_corr <= threshold or np.isnan(max_corr), \
    f'Found remaining pair with correlation {max_corr:.3f} > {threshold}'

print('Exercițiul 9 a fost rezolvat corect!')
print(f'\nSetul de date Wine ({X_wine_df.shape[1]} caracteristici) → păstrate {len(kept)}, eliminate {len(removed)}')
print(f'Eliminate: {removed}')


In [ ]:
unittests.exercise_9(remove_correlated_features)

<a name='4'></a>
## 4 - Pipeline de Selecție a Caracteristicilor

Folosirea unui `Pipeline` asigură că selectorul este antrenat **doar pe datele de antrenare**, prevenind scurgerea de informații din setul de test.

Pipeline-ul va lega pașii: `StandardScaler → SelectKBest → LogisticRegression`


<a name='4-1'></a>
### 4.1 - Construiește Pipeline-ul Complet

<a name='ex-10'></a>
**Exercițiul 10 - create_selection_pipeline**

Construiește și returnează un `Pipeline` sklearn cu trei pași:
1. `('scaler', StandardScaler())`
2. `('selector', SelectKBest(score_func, k=k))`
3. `('model', LogisticRegression(max_iter=1000))`

**Instrucțiuni:**
- Returnează un obiect Pipeline **neantrenat**
- Folosește argumentele `k` și `score_func` ca parametri


In [ ]:
def create_selection_pipeline(k, score_func=f_classif):
    """
    Construiește un pipeline de selecție a caracteristicilor:
    StandardScaler -> SelectKBest -> LogisticRegression

    Args:
        k: Numărul de caracteristici de selectat
        score_func: Funcția de scor pentru SelectKBest (implicit: f_classif)

    Returns:
        pipeline: Pipeline sklearn neantrenat
    """
    ### ÎNCEPUT DE COD AICI ### (~6 lines)
    pipeline = None
    ### SFÂRȘIT DE COD AICI ###

    return pipeline


In [ ]:
# Test pentru create_selection_pipeline
pipe = create_selection_pipeline(k=10, score_func=f_classif)

assert pipe is not None, 'Pipeline must not be None'
assert hasattr(pipe, 'fit'), 'Return value must be a Pipeline'
assert 'scaler' in pipe.named_steps, 'Pipeline must contain a scaler step'
assert 'selector' in pipe.named_steps, 'Pipeline must contain a selector step'
assert 'model' in pipe.named_steps, 'Pipeline must contain a model step'

# Aplică fit și evaluează pentru a verifica fluxul cap-coadă
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
pipe.fit(X_tr, y_tr)
accuracy = pipe.score(X_te, y_te)

print('Exercițiul 10 a fost rezolvat corect!')
print(f'Acuratețea pipeline-ului pe test: {accuracy:.3f}')

# Compară valorile k folosind validare încrucișată
k_values = [5, 8, 10, 15, 20, 25, 30]
cv_scores = []
for k_val in k_values:
    p = create_selection_pipeline(k=k_val)
    score = cross_val_score(p, X_clf, y_clf, cv=5, scoring='accuracy').mean()
    cv_scores.append(score)

plt.figure(figsize=(8, 4))
plt.plot(k_values, cv_scores, marker='o', color='steelblue')
plt.xlabel('Numărul de caracteristici selectate (k)')
plt.ylabel('Acuratețe CV în 5 fold-uri')
plt.title('Numărul de caracteristici vs performanța modelului')
plt.xticks(k_values)
plt.tight_layout()
plt.show()


In [ ]:
unittests.exercise_10(create_selection_pipeline)

<a name='5'></a>
## 5 - Implementare de la Zero

Implementarea de la zero a statisticii F ANOVA îți construiește o intuiție mai profundă despre ceea ce face intern `f_classif`.

**Reamintește-ți formula:**
$$F_j = \frac{\text{varianța între grupuri a caracteristicii } j}{\text{varianța în interiorul grupurilor a caracteristicii } j}$$

unde:
- **Varianța între grupuri** = $\sum_c n_c (\bar{x}_{c,j} - \bar{x}_j)^2 \;/\; (K-1)$
- **Varianța în interiorul grupurilor** = $\sum_c \sum_{i \in c} (x_{i,j} - \bar{x}_{c,j})^2 \;/\; (N-K)$

($K$ = numărul de clase, $N$ = numărul de eșantioane, $\bar{x}_{c,j}$ = media caracteristicii $j$ în clasa $c$)


<a name='5-1'></a>
### 5.1 - Statistica F ANOVA de la Zero

<a name='ex-bonus'></a>
**Exercițiu Bonus - compute_f_statistic_from_scratch**

Implementează calculul statisticii F ANOVA pentru fiecare caracteristică, independent.

**Instrucțiuni:**
- Calculează media globală a fiecărei caracteristici pe toate eșantioanele (`overall_mean`)
- Pentru fiecare clasă `c`, calculează `class_mean_c` (media fiecărei caracteristici în clasa `c`) și `n_c` (numărul de eșantioane)
- Calculează varianța între grupuri și varianța în interiorul grupurilor folosind formulele de mai sus
- Returnează un array de scoruri F, unul pentru fiecare caracteristică


In [ ]:
def compute_f_statistic_from_scratch(X, y):
    """
    Calculează manual statistica F ANOVA pentru fiecare caracteristică.

    Args:
        X: Matrice de caracteristici de formă (n_samples, n_features), cu valori continue
        y: Etichete întregi de clasă de formă (n_samples,)

    Returns:
        f_scores: array NumPy cu scoruri F, de formă (n_features,)
    """
    n_samples, n_features = X.shape
    classes = np.unique(y)
    K = len(classes)

    ### ÎNCEPUT DE COD AICI ### (~15 lines)
    f_scores = None
    ### SFÂRȘIT DE COD AICI ###

    return f_scores


In [ ]:
# Test pentru compute_f_statistic_from_scratch
f_manual = compute_f_statistic_from_scratch(X_clf_scaled, y_clf)

if f_manual is not None:
    # Compară cu f_classif din sklearn
    f_sklearn, _ = f_classif(X_clf_scaled, y_clf)

    assert f_manual.shape == (X_clf_scaled.shape[1],), \
        f'Expected shape ({X_clf_scaled.shape[1]},), got {f_manual.shape}'
    assert np.allclose(f_manual, f_sklearn, rtol=1e-3), \
        'F-scores do not match sklearn. Check your formula.'

    print('Exercițiul bonus a fost rezolvat corect!')
    print(f'Eroarea absolută maximă față de sklearn: {np.max(np.abs(f_manual - f_sklearn)):.6f}')
else:
    print('Bonusul nu este încă implementat — se sare peste comparație')


In [ ]:
unittests.exercise_11(compute_f_statistic_from_scratch)

<a name='6'></a>
## 6 - Raport de Selecție a Caracteristicilor

Documentează procesul tău de selecție a caracteristicilor completând șablonul de mai jos.

Un raport bun răspunde la:
1. Cu câte caracteristici ai **pornit**?
2. Ce caracteristici au fost **eliminate** la fiecare pas și de ce?
3. Care a fost **efectul asupra performanței modelului**?

---

### Raportul Tău

*(Editează această celulă pentru a completa concluziile tale)*

**Set de date:** Clasificare sintetică (30 caracteristici, 600 eșantioane)

**Pasul 1 – Prag de Varianță**
- Prag folosit: `___`
- Caracteristici eliminate: `___`
- Motiv: ...

**Pasul 2 – Test Statistic (f_classif)**
- k selectat: `___`
- Primele 3 caracteristici (după F-score): `___`
- Caracteristici cu p > 0.05: `___`

**Pasul 3 – Eliminare pe baza Corelației**
- Prag folosit: `___`
- Perechi găsite: `___`
- Caracteristici eliminate: `___`

**Comparația Modelelor**

| Configurație | Acuratețe CV în 5 fold-uri |
|---|---|
| Toate cele 30 de caracteristici | ??? |
| După selecția caracteristicilor | ??? |

**Concluzie:** ...
